In [2]:
import pandas as pd
import numpy as np
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
# rumus learning curve
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

def plot_my_learning_curve(model, X, y, title="Learning Curve"):
    plt.figure(figsize=(8, 5))
    
    # Hitung learning curve
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=5, scoring='r2', n_jobs=-1, 
        train_sizes=np.linspace(0.1, 1.0, 5), random_state=42
    )
    
    # Hitung rata-rata dan standar deviasi
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)
    
    # Gambar grafik
    plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, test_mean, 'o-', color="g", label="Cross-validation score")
    
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="g")
    
    plt.title(title)
    plt.xlabel("Training Examples")
    plt.ylabel("R2 Score")
    plt.legend(loc="best")
    plt.grid(True)
    plt.show()

# CARA PAKAI (Jalankan setelah kamu load X dan y Supplies):
# plot_my_learning_curve(best_supplies_model, X_full, y_full, title="Learning Curve - Extra Trees Supplies")

In [10]:

# =========================================================================
# 1. PREPARASI DATA & PEMBERSIHAN DATA LEAKAGE
# =========================================================================
# Membaca file data proses bulanan
df = pd.read_csv(r'D:\Sem 4\PBL\data\processed\data_proses_bulanan.csv')

# Pisahkan Target (Quantity) dan Fitur
# Kita buang 'Order Date', 'Sales', dan 'Profit' (Data bulan berjalan / leakage)
X = df.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity'])
y = df['Quantity']

# Karena kita tidak pakai PyCaret, kita harus ubah kolom teks 'Sub-Category' 
# menjadi angka (One-Hot Encoding) secara manual agar bisa dibaca Sklearn & LightGBM
X = pd.get_dummies(X, columns=['Sub-Category'], drop_first=True)

# Split data menjadi Train Set (80%) dan Test Set (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data Berhasil Disiapkan!")
print(f"Jumlah Data Training: {X_train.shape[0]} baris")
print(f"Jumlah Data Testing : {X_test.shape[0]} baris\n")

# =========================================================================
# 2. FUNGSI EVALUASI CUSTOM (Menghindari Error Division by Zero pada MAPE)
# =========================================================================
def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Mask untuk menghindari pembagian dengan angka 0 jika ada quantity bernilai 0
    mask = y_true != 0
    if np.sum(mask) == 0:
        return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

# =========================================================================
# 3. DEFINISI TOP 3 BASELINE MODEL SECARA NATIVE
# =========================================================================
models = {
    "Extra_Trees_Regressor": ExtraTreesRegressor(
        n_estimators=100,      # Sesuaikan dengan nilai dari PyCaret
        max_depth=10,          # Sesuaikan dengan nilai dari PyCaret
        min_samples_split=2,   # Sesuaikan dengan nilai dari PyCaret
        random_state=42, 
        n_jobs=-1
    ),
    
    "Gradient_Boosting_Regressor": GradientBoostingRegressor(
        n_estimators=150, 
        learning_rate=0.1, 
        max_depth=5,
        random_state=42
    ),
    
    "LightGBM_Regressor": LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42, 
        verbose=-1,
        n_jobs=-1
    )
}

# =========================================================================
# 4. LOOPING MODELING MANUAL & TRACKING DENGAN MLFLOW
# =========================================================================
# Tambahkan baris ini untuk MEMPAKSA Python menulis ke folder proyekmu yang benar
mlflow.set_tracking_uri(r"file:///D:/Sem 4/PBL/model-pycaret/mlruns")  # Ganti dengan path folder mlruns di proyekmu
# Tentukan nama eksperimen induk di MLflow
mlflow.set_experiment("Global_Monthly_Manual_Modeling")

print("Mulai melakukan looping training dan manual tracking ke MLflow...")

for model_name, model_obj in models.items():
    # Membuat 'Run' terpisah untuk masing-masing dari Top 3 Model di MLflow
    with mlflow.start_run(run_name=model_name):
        print(f"\n[RUNNING] Melatih Model: {model_name}...")
        
        # Training Model secara manual
        model_obj.fit(X_train, y_train)
        
        # Prediksi menggunakan Test Set
        y_pred = model_obj.predict(X_test)
        
        # Hitung Metrik Evaluasi sesuai dengan tabel dokumentasimu
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        mape = calculate_mape(y_test, y_pred)
        
        # --- TRACKING MANUAL KE MLFLOW ---
        # 1. Log Parameter Model (Otomatis merekam settingan hyperparameter bawaan)
        mlflow.log_params(model_obj.get_params())
        
        # 2. Log Metrik Evaluasi Hasil Pengujian
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)
        mlflow.log_metric("MAPE", mape)
        
        # 3. Log File Model (.pkl) ke dalam MLflow Artifacts
        mlflow.sklearn.log_model(model_obj, artifact_path="model_file")
        
        # Cetak hasil ke terminal/notebook untuk pembuktian instan
        print(f"✨ {model_name} Berhasil Tercatat!")
        print(f"   |-- MAE  : {mae:.4f}")
        print(f"   |-- RMSE : {rmse:.4f}")
        print(f"   |-- R2   : {r2:.4f}")
        print(f"   |-- MAPE : {mape:.4f}")

print("\n=======================================================")
print("✅ SELESAI! Top 3 Model berhasil dilatih & di-track ke MLflow!")
print("Silakan ketik 'mlflow ui' di terminal untuk melihat hasilnya.")
print("=======================================================")

Data Berhasil Disiapkan!
Jumlah Data Training: 71 baris
Jumlah Data Testing : 18 baris

Mulai melakukan looping training dan manual tracking ke MLflow...

[RUNNING] Melatih Model: Extra_Trees_Regressor...
✨ Extra_Trees_Regressor Berhasil Tercatat!
   |-- MAE  : 4.6953
   |-- RMSE : 5.6570
   |-- R2   : 0.0835
   |-- MAPE : 0.6224

[RUNNING] Melatih Model: Gradient_Boosting_Regressor...
✨ Gradient_Boosting_Regressor Berhasil Tercatat!
   |-- MAE  : 5.2155
   |-- RMSE : 6.4558
   |-- R2   : -0.1936
   |-- MAPE : 0.7475

[RUNNING] Melatih Model: LightGBM_Regressor...
✨ LightGBM_Regressor Berhasil Tercatat!
   |-- MAE  : 4.1415
   |-- RMSE : 5.4568
   |-- R2   : 0.1472
   |-- MAPE : 0.5313

✅ SELESAI! Top 3 Model berhasil dilatih & di-track ke MLflow!
Silakan ketik 'mlflow ui' di terminal untuk melihat hasilnya.


# Tuning LightGboost dan Extra Tree Regression


In [15]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import ExtraTreesRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Matikan log bising dari Optuna agar terminal tetap bersih
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =========================================================================
# 1. PREPARASI DATA (Sama seperti sebelumnya)
# =========================================================================
df = pd.read_csv(r'D:\Sem 4\PBL\data\processed\data_proses_bulanan.csv')
X = df.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity'])
y = df['Quantity']
X = pd.get_dummies(X, columns=['Sub-Category'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) if np.sum(mask) > 0 else 0.0

# Set Eksperimen induk di MLflow
mlflow.set_tracking_uri("file:///D:/Sem 4/PBL/model-pycaret/mlruns")
mlflow.set_experiment("Global_Monthly_Manual_Modeling")

# =========================================================================
# 2. TUNING EXTRA TREES REGRESSOR
# =========================================================================
def objective_et(trial):
    # Tentukan ruang pencarian parameter (Search Space)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': 42,
        'n_jobs': -1
    }
    
    # Hitung validasi dalam menggunakan 3-Fold CV agar tidak overfit ke data test
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = ExtraTreesRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(mean_absolute_error(y_val, preds))
        
    return np.mean(scores) # Goal Optuna: Meminimalkan nilai MAE ini

print("-> Memulai Tuning Optuna untuk Extra Trees (50 Trials)...")
study_et = optuna.create_study(direction='minimize')
study_et.optimize(objective_et, n_trials=50)

# Catat Hasil Terbaik Extra Trees ke MLflow
with mlflow.start_run(run_name="Tuned_Extra_Trees"):
    print("[LOGGING] Mencatat Tuned Extra Trees ke MLflow...")
    
    # Train ulang dengan seluruh data training menggunakan parameter terbaik
    best_params_et = study_et.best_params
    best_model_et = ExtraTreesRegressor(**best_params_et, random_state=42, n_jobs=-1)
    best_model_et.fit(X_train, y_train)
    
    # Uji ke data test asli
    y_pred_et = best_model_et.predict(X_test)
    
    # Hitung metrik akhir
    mae_et = mean_absolute_error(y_test, y_pred_et)
    rmse_et = np.sqrt(mean_squared_error(y_test, y_pred_et))
    r2_et = r2_score(y_test, y_pred_et)
    mape_et = calculate_mape(y_test, y_pred_et)
    
    # Kirim ke MLflow
    mlflow.log_params(best_params_et)
    mlflow.log_metric("MAE", mae_et)
    mlflow.log_metric("RMSE", rmse_et)
    mlflow.log_metric("R2", r2_et)
    mlflow.log_metric("MAPE", mape_et)
    mlflow.sklearn.log_model(best_model_et, artifact_path="model_tuned_et")
    print(f"✨ Tuned ET Done! R2 Akhir: {r2_et:.4f} | MAE: {mae_et:.4f}")


# =========================================================================
# 3. TUNING LIGHTGBM REGRESSOR
# =========================================================================
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 2, 20),
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(mean_absolute_error(y_val, preds))
        
    return np.mean(scores)

print("\n-> Memulai Tuning Optuna untuk LightGBM (50 Trials)...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=50)

# Catat Hasil Terbaik LightGBM ke MLflow
with mlflow.start_run(run_name="Tuned_LightGBM"):
    print("[LOGGING] Mencatat Tuned LightGBM ke MLflow...")
    
    best_params_lgb = study_lgb.best_params
    best_model_lgb = LGBMRegressor(**best_params_lgb, random_state=42, verbose=-1, n_jobs=-1)
    best_model_lgb.fit(X_train, y_train)
    
    y_pred_lgb = best_model_lgb.predict(X_test)
    
    mae_lgb = mean_absolute_error(y_test, y_pred_lgb)
    rmse_lgb = np.sqrt(mean_squared_error(y_test, y_pred_lgb))
    r2_lgb = r2_score(y_test, y_pred_lgb)
    mape_lgb = calculate_mape(y_test, y_pred_lgb)
    
    mlflow.log_params(best_params_lgb)
    mlflow.log_metric("MAE", mae_lgb)
    mlflow.log_metric("RMSE", rmse_lgb)
    mlflow.log_metric("R2", r2_lgb)
    mlflow.log_metric("MAPE", mape_lgb)
    mlflow.sklearn.log_model(best_model_lgb, artifact_path="model_tuned_lgb")
    print(f"✨ Tuned LGBM Done! R2 Akhir: {r2_lgb:.4f} | MAE: {mae_lgb:.4f}")

print("\n=======================================================")
# Perbaikan teks typo instruksi user
print("✅ SINKRONISASI SELESAI! Hasil tuning sukses direkam.")
print("Buka MLflow UI untuk membandingkan run 'Tuned_...' dengan versi baseline kemarin.")
print("=======================================================")

-> Memulai Tuning Optuna untuk Extra Trees (50 Trials)...
[LOGGING] Mencatat Tuned Extra Trees ke MLflow...
✨ Tuned ET Done! R2 Akhir: 0.1178 | MAE: 4.3076

-> Memulai Tuning Optuna untuk LightGBM (50 Trials)...
[LOGGING] Mencatat Tuned LightGBM ke MLflow...
✨ Tuned LGBM Done! R2 Akhir: -0.2932 | MAE: 5.3119

✅ SINKRONISASI SELESAI! Hasil tuning sukses direkam.
Buka MLflow UI untuk membandingkan run 'Tuned_...' dengan versi baseline kemarin.


# finalisasi model

In [ ]:
# import pickle
# import os

# print("=======================================================")
# print("🚀 MEMULAI PROSES FINALISASI MODEL PEMENANG")
# print("=======================================================")

# # 1. Ambil hyperparameter terbaik dari objek study_et yang sudah selesai running tadi
# is_study_exist = 'study_et' in locals() or 'study_et' in globals()

# if is_study_exist:
#     best_params = study_et.best_params
#     print(f"Menggunakan Parameter Terbaik Optuna: {best_params}")
# else:
#     # Backup plan jika kamu tidak sengaja merestart kernel notebook
#     best_params = {
#         'n_estimators': 150,
#         'max_depth': 12,
#         'min_samples_split': 2,
#         'min_samples_leaf': 1
#     }
#     print("Membaca parameter backup (Pastikan runtime notebook tidak terputus).")

# # 2. Definisikan model final menggunakan Extra Trees
# model_final_bulanan = ExtraTreesRegressor(**best_params, random_state=42, n_jobs=-1)

# # 3. Latih ke SELURUH DATA (100% data gabungan X dan y)
# # Ini adalah standar industri agar model siap memprediksi masa depan dengan data maksimal
# print("-> Melatih model Extra Trees ke 100% data master bulanan...")
# model_final_bulanan.fit(X, y)

# # 4. Simpan model menjadi file .pkl (Pickle)
# os.makedirs('models', exist_ok=True)
# path_simpan = 'models/model_monthly_global.pkl'

# with open(path_simpan, 'wb') as f:
#     pickle.dump(model_final_bulanan, f)

# print(f"✅ BERHASIL! Model matang disimpan di: {path_simpan}")
# print("=======================================================")

🚀 MEMULAI PROSES FINALISASI MODEL PEMENANG
Menggunakan Parameter Terbaik Optuna: {'n_estimators': 71, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 6}
-> Melatih model Extra Trees ke 100% data master bulanan...
✅ BERHASIL! Model matang disimpan di: models/model_monthly_global.pkl


# learning curve

In [1]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Impor Model Regresi secara Native
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor

# =========================================================================
# 1. PREPARASI DATA & PEMBERSIHAN DATA LEAKAGE
# =========================================================================
df = pd.read_csv(r'D:\Sem 4\PBL\data\processed\data_proses_bulanan.csv')

# Pisahkan Target (Quantity) dan Fitur
X = df.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity'])
y = df['Quantity']

# One-Hot Encoding manual untuk kolom kategori 'Sub-Category'
X = pd.get_dummies(X, columns=['Sub-Category'], drop_first=True)

# Split data menjadi Train Set (80%) dan Test Set (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Data Berhasil Disiapkan!")
print(f"Jumlah Data Training: {X_train.shape[0]} baris")
print(f"Jumlah Data Testing : {X_test.shape[0]} baris\n")

# =========================================================================
# 2. FUNGSI EVALUASI CUSTOM
# =========================================================================
def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if np.sum(mask) == 0:
        return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

# =========================================================================
# 3. DEFINISI TOP 3 BASELINE MODEL SECARA NATIVE
# =========================================================================
models = {
    "Extra_Trees_Regressor": ExtraTreesRegressor(
        n_estimators=100,      
        max_depth=10,          
        min_samples_split=2,   
        random_state=42, 
        n_jobs=-1
    ),
    
    "Gradient_Boosting_Regressor": GradientBoostingRegressor(
        n_estimators=150, 
        learning_rate=0.1, 
        max_depth=5,
        random_state=42
    ),
    
    "LightGBM_Regressor": LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42, 
        verbose=-1,
        n_jobs=-1
    )
}

# =========================================================================
# 4. LOOPING MODELING MANUAL & TRACKING DENGAN MLFLOW + LEARNING CURVE
# =========================================================================
mlflow.set_tracking_uri(r"file:///D:/Sem 4/PBL/model-pycaret/mlruns")  
mlflow.set_experiment("Global_Monthly_Manual_Modeling")

print("🚀 Mulai melakukan looping training, evaluasi, dan pembuatan Learning Curve...")

for model_name, model_obj in models.items():
    with mlflow.start_run(run_name=model_name):
        print(f"\n[RUNNING] Melatih Model: {model_name}...")
        
        # Training Model secara manual
        model_obj.fit(X_train, y_train)
        
        # Prediksi menggunakan Test Set
        y_pred = model_obj.predict(X_test)
        
        # Hitung Metrik Evaluasi
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        mape = calculate_mape(y_test, y_pred)
        
        # -----------------------------------------------------------------
        # TAHAP PRO: PROSES PEMBUATAN & PLOTTING LEARNING CURVE
        # -----------------------------------------------------------------
        print(f"📊 Menghitung Learning Curve untuk {model_name}...")
        
        # Menghitung skor cross-validation seiring bertambahnya ukuran data training
        # Menggunakan scoring 'r2' dengan 5-Fold Cross Validation
        train_sizes, train_scores, valid_scores = learning_curve(
            model_obj, X_train, y_train, cv=5, scoring='r2', 
            train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1, random_state=42
        )
        
        # Hitung rata-rata skor dari hasil cross-validation
        train_mean = np.mean(train_scores, axis=1)
        valid_mean = np.mean(valid_scores, axis=1)
        
        # Gambar grafik menggunakan Matplotlib
        plt.figure(figsize=(8, 5))
        plt.plot(train_sizes, train_mean, 'o-', color="red", label="Training Score ($R^2$)")
        plt.plot(train_sizes, valid_mean, 'o-', color="green", label="Cross-Validation Score ($R^2$)")
        
        plt.title(f"Learning Curve: {model_name.replace('_', ' ')}")
        plt.xlabel("Ukuran Data Training (Baris)")
        plt.ylabel("Skor $R^2$")
        plt.legend(loc="best")
        plt.grid(True)
        
        # Simpan grafik sebagai file gambar lokal sementara
        plot_filename = f"{model_name}_learning_curve.png"
        plt.savefig(plot_filename, bbox_inches='tight')
        plt.close() # Menutup plot agar tidak menumpuk di memori RAM notebook
        
        # -----------------------------------------------------------------
        # TRACKING MANUAL KE MLFLOW
        # -----------------------------------------------------------------
        # 1. Log Parameter Model
        mlflow.log_params(model_obj.get_params())
        
        # 2. Log Metrik Evaluasi
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)
        mlflow.log_metric("MAPE", mape)
        
        # 3. Log File Gambar Learning Curve (.png) ke dalam MLflow Artifacts
        mlflow.log_artifact(plot_filename, artifact_path="evaluation_plots")
        
        # 4. Log File Model (.pkl) ke dalam MLflow Artifacts
        mlflow.sklearn.log_model(model_obj, artifact_path="model_file")
        
        # Hapus file gambar lokal sementara agar folder proyek tetap bersih
        if os.path.exists(plot_filename):
            os.remove(plot_filename)
        
        # Cetak hasil ke terminal
        print(f"✨ {model_name} Berhasil Tercatat!")
        print(f"   |-- MAE  : {mae:.4f}")
        print(f"   |-- RMSE : {rmse:.4f}")
        print(f"   |-- R2   : {r2:.4f}")
        print(f"   |-- MAPE : {mape:.4f}")
        print(f"   |-- [INFO] Grafik Learning Curve sukses diunggah ke MLflow Artifacts!")

print("\n=======================================================")
print("✅ SELESAI! Top 3 Model & Learning Curves berhasil di-track!")
print("Silakan buka terminal, ketik 'mlflow ui', lalu cek tab 'Artifacts' pada setiap Run.")
print("=======================================================")

✅ Data Berhasil Disiapkan!
Jumlah Data Training: 71 baris
Jumlah Data Testing : 18 baris

🚀 Mulai melakukan looping training, evaluasi, dan pembuatan Learning Curve...

[RUNNING] Melatih Model: Extra_Trees_Regressor...
📊 Menghitung Learning Curve untuk Extra_Trees_Regressor...
✨ Extra_Trees_Regressor Berhasil Tercatat!
   |-- MAE  : 4.6953
   |-- RMSE : 5.6570
   |-- R2   : 0.0835
   |-- MAPE : 0.6224
   |-- [INFO] Grafik Learning Curve sukses diunggah ke MLflow Artifacts!

[RUNNING] Melatih Model: Gradient_Boosting_Regressor...
📊 Menghitung Learning Curve untuk Gradient_Boosting_Regressor...
✨ Gradient_Boosting_Regressor Berhasil Tercatat!
   |-- MAE  : 5.2155
   |-- RMSE : 6.4558
   |-- R2   : -0.1936
   |-- MAPE : 0.7475
   |-- [INFO] Grafik Learning Curve sukses diunggah ke MLflow Artifacts!

[RUNNING] Melatih Model: LightGBM_Regressor...
📊 Menghitung Learning Curve untuk LightGBM_Regressor...
✨ LightGBM_Regressor Berhasil Tercatat!
   |-- MAE  : 4.1415
   |-- RMSE : 5.4568
   |-- 

In [3]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import optuna
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, learning_curve
from sklearn.ensemble import ExtraTreesRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Matikan log bising dari Optuna agar terminal tetap bersih
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =========================================================================
# 1. PREPARASI DATA (Sama seperti sebelumnya)
# =========================================================================
df = pd.read_csv(r'D:\Sem 4\PBL\data\processed\data_proses_bulanan.csv')
X = df.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity'])
y = df['Quantity']
X = pd.get_dummies(X, columns=['Sub-Category'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) if np.sum(mask) > 0 else 0.0

# Set Eksperimen induk di MLflow
mlflow.set_tracking_uri("file:///D:/Sem 4/PBL/model-pycaret/mlruns")
mlflow.set_experiment("Global_Monthly_Manual_Modeling")


# =========================================================================
# 2. TUNING EXTRA TREES REGRESSOR
# =========================================================================
def objective_et(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': 42,
        'n_jobs': -1
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = ExtraTreesRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(mean_absolute_error(y_val, preds))
        
    return np.mean(scores)

print("-> Memulai Tuning Optuna untuk Extra Trees (50 Trials)...")
study_et = optuna.create_study(direction='minimize')
study_et.optimize(objective_et, n_trials=50)

# Catat Hasil Terbaik Extra Trees ke MLflow
with mlflow.start_run(run_name="Tuned_Extra_Trees"):
    print("[LOGGING] Mencatat Tuned Extra Trees ke MLflow...")
    
    # Train ulang dengan seluruh data training menggunakan parameter terbaik
    best_params_et = study_et.best_params
    best_model_et = ExtraTreesRegressor(**best_params_et, random_state=42, n_jobs=-1)
    best_model_et.fit(X_train, y_train)
    
    # Uji ke data test asli
    y_pred_et = best_model_et.predict(X_test)
    
    # Hitung metrik akhir
    mae_et = mean_absolute_error(y_test, y_pred_et)
    rmse_et = np.sqrt(mean_squared_error(y_test, y_pred_et))
    r2_et = r2_score(y_test, y_pred_et)
    mape_et = calculate_mape(y_test, y_pred_et)
    
    # -----------------------------------------------------------------
    # VISUALISASI LEARNING CURVE - TUNED EXTRA TREES
    # -----------------------------------------------------------------
    print("📊 Menghitung Learning Curve untuk Tuned Extra Trees...")
    train_sizes_et, train_scores_et, valid_scores_et = learning_curve(
        best_model_et, X_train, y_train, cv=5, scoring='r2', 
        train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1, random_state=42
    )
    
    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes_et, np.mean(train_scores_et, axis=1), 'o-', color="red", label="Training Score ($R^2$)")
    plt.plot(train_sizes_et, np.mean(valid_scores_et, axis=1), 'o-', color="green", label="Cross-Validation Score ($R^2$)")
    plt.title("Learning Curve: Tuned Extra Trees")
    plt.xlabel("Ukuran Data Training (Baris)")
    plt.ylabel("Skor $R^2$")
    plt.legend(loc="best")
    plt.grid(True)
    
    plot_et_name = "tuned_extra_trees_learning_curve.png"
    plt.savefig(plot_et_name, bbox_inches='tight')
    plt.close()
    
    # Kirim ke MLflow
    mlflow.log_params(best_params_et)
    mlflow.log_metric("MAE", mae_et)
    mlflow.log_metric("RMSE", rmse_et)
    mlflow.log_metric("R2", r2_et)
    mlflow.log_metric("MAPE", mape_et)
    mlflow.log_artifact(plot_et_name, artifact_path="evaluation_plots")
    mlflow.sklearn.log_model(best_model_et, artifact_path="model_tuned_et")
    
    # Bersihkan file gambar lokal
    if os.path.exists(plot_et_name):
        os.remove(plot_et_name)
        
    print(f"✨ Tuned ET Done! R2 Akhir: {r2_et:.4f} | MAE: {mae_et:.4f}")
    print("   [INFO] Grafik Learning Curve sukses diunggah!")


# =========================================================================
# 3. TUNING LIGHTGBM REGRESSOR
# =========================================================================
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 2, 20),
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(mean_absolute_error(y_val, preds))
        
    return np.mean(scores)

print("\n-> Memulai Tuning Optuna untuk LightGBM (50 Trials)...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=50)

# Catat Hasil Terbaik LightGBM ke MLflow
with mlflow.start_run(run_name="Tuned_LightGBM"):
    print("[LOGGING] Mencatat Tuned LightGBM ke MLflow...")
    
    best_params_lgb = study_lgb.best_params
    best_model_lgb = LGBMRegressor(**best_params_lgb, random_state=42, verbose=-1, n_jobs=-1)
    best_model_lgb.fit(X_train, y_train)
    
    y_pred_lgb = best_model_lgb.predict(X_test)
    
    mae_lgb = mean_absolute_error(y_test, y_pred_lgb)
    rmse_lgb = np.sqrt(mean_squared_error(y_test, y_pred_lgb))
    r2_lgb = r2_score(y_test, y_pred_lgb)
    mape_lgb = calculate_mape(y_test, y_pred_lgb)
    
    # -----------------------------------------------------------------
    # VISUALISASI LEARNING CURVE - TUNED LIGHTGBM
    # -----------------------------------------------------------------
    print("📊 Menghitung Learning Curve untuk Tuned LightGBM...")
    train_sizes_lgb, train_scores_lgb, valid_scores_lgb = learning_curve(
        best_model_lgb, X_train, y_train, cv=5, scoring='r2', 
        train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1, random_state=42
    )
    
    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes_lgb, np.mean(train_scores_lgb, axis=1), 'o-', color="red", label="Training Score ($R^2$)")
    plt.plot(train_sizes_lgb, np.mean(valid_scores_lgb, axis=1), 'o-', color="green", label="Cross-Validation Score ($R^2$)")
    plt.title("Learning Curve: Tuned LightGBM")
    plt.xlabel("Ukuran Data Training (Baris)")
    plt.ylabel("Skor $R^2$")
    plt.legend(loc="best")
    plt.grid(True)
    
    plot_lgb_name = "tuned_lightgbm_learning_curve.png"
    plt.savefig(plot_lgb_name, bbox_inches='tight')
    plt.close()
    
    # Kirim ke MLflow
    mlflow.log_params(best_params_lgb)
    mlflow.log_metric("MAE", mae_lgb)
    mlflow.log_metric("RMSE", rmse_lgb)
    mlflow.log_metric("R2", r2_lgb)
    mlflow.log_metric("MAPE", mape_lgb)
    mlflow.log_artifact(plot_lgb_name, artifact_path="evaluation_plots")
    mlflow.sklearn.log_model(best_model_lgb, artifact_path="model_tuned_lgb")
    
    # Bersihkan file gambar lokal
    if os.path.exists(plot_lgb_name):
        os.remove(plot_lgb_name)
        
    print(f"✨ Tuned LGBM Done! R2 Akhir: {r2_lgb:.4f} | MAE: {mae_lgb:.4f}")
    print("   [INFO] Grafik Learning Curve sukses diunggah!")

print("\n=======================================================")
print("✅ SINKRONISASI SELESAI! Hasil tuning & plot sukses direkam.")
print("Buka MLflow UI untuk melihat performa parameter terbaik beserta visualisasinya.")
print("=======================================================")

-> Memulai Tuning Optuna untuk Extra Trees (50 Trials)...
[LOGGING] Mencatat Tuned Extra Trees ke MLflow...
📊 Menghitung Learning Curve untuk Tuned Extra Trees...
✨ Tuned ET Done! R2 Akhir: 0.1246 | MAE: 4.3251
   [INFO] Grafik Learning Curve sukses diunggah!

-> Memulai Tuning Optuna untuk LightGBM (50 Trials)...
[LOGGING] Mencatat Tuned LightGBM ke MLflow...
📊 Menghitung Learning Curve untuk Tuned LightGBM...
✨ Tuned LGBM Done! R2 Akhir: -0.2091 | MAE: 5.0199
   [INFO] Grafik Learning Curve sukses diunggah!

✅ SINKRONISASI SELESAI! Hasil tuning & plot sukses direkam.
Buka MLflow UI untuk melihat performa parameter terbaik beserta visualisasinya.
